# Pass 1 — Schnelltest

Fuehrt nur Pass 1 durch — ohne Anonymisierung, ohne ChromaDB, ohne Pass 0.  
Nutzt die Artefakte aus `data/debug_last_run/` direkt.

**Artefakte:**
- `02_anonymized_text.txt` — anonymisierter Text fuer Pass 1
- `06_final_result.json` — letztes Gesamtergebnis zum Vergleich

## 1 — Setup

In [ ]:
import sys, json, re, os
from pathlib import Path
import pandas as pd

sys.path.insert(0, "../src")

from news_analyser.prompts import load_prompt
import llm_adapter

DEBUG   = Path("../data/debug_last_run")
adapter = llm_adapter.get_instance(os.environ.get("LLM_PROVIDER", "openai"))

print(f"Provider: {adapter.name}  |  Modell: {adapter.model}")

## 2 — Artefakte laden

In [ ]:
anon_text  = (DEBUG / "02_anonymized_text.txt").read_text(encoding="utf-8").strip()
word_count = len(anon_text.split())

print(f"{word_count} Woerter im anonymisierten Text")
print("---")
print(anon_text[:400], "...")

## 3 — Pass 1 ausfuehren

In [ ]:
pass1_input = {
    "source_url":  "local://debug_last_run",
    "domain":      "debug",
    "word_count":  word_count,
    "keyword_signal": {
        "extremism_score": 0.0, "left_count": 0, "right_count": 0, "general_count": 0,
        "left_hits": [], "right_hits": [], "general_hits": []
    },
    "text": anon_text,
}

pass1_prompt = load_prompt("system", "pass1")
print("Pass 1 laeuft ...", flush=True)
raw = adapter.generate(system_prompt=pass1_prompt, input_data=pass1_input)
print("Fertig.")

## 4 — Ergebnis parsen

In [ ]:
def extract_json(raw):
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        try:
            from json_repair import repair_json
            return json.loads(repair_json(cleaned))
        except Exception as e:
            print(f"Parse-Fehler: {e}")
            print("Raw:", raw[:500])
            return {}

result     = extract_json(raw)
techniques = result.get("detected_techniques", [])
ft         = result.get("framing_target", {})
bernays    = round(len(techniques) / max(word_count, 1) * 1000, 2)

print(f"Orwell-Index:  {ft.get('orwell_index', '-')}")
print(f"Bernays Score: {bernays}  ({len(techniques)} Instanzen / {word_count} Woerter)")
if result.get("symmetry_note"):
    print(f"Symmetrie:     {result['symmetry_note']}")

## 5 — Techniken als Tabelle

In [ ]:
if not techniques:
    print("Keine Techniken erkannt.")
else:
    rows = [
        {
            "#": i + 1,
            "Technik": t["technique"],
            "Zitat": t.get("quote", "")[:80] + ("..." if len(t.get("quote", "")) > 80 else ""),
            "Erklaerung": t.get("explanation", "")
        }
        for i, t in enumerate(techniques)
    ]
    df = pd.DataFrame(rows).set_index("#")
    pd.set_option("display.max_colwidth", 90)
    display(df)

## 6 — Vergleich mit letztem Gesamtergebnis

In [ ]:
prev_file = DEBUG / "06_final_result.json"
if prev_file.exists():
    prev         = json.loads(prev_file.read_text(encoding="utf-8"))
    prev_tech    = prev.get("detected_techniques", [])
    prev_orwell  = prev.get("framing_target", {}).get("orwell_index", "-")
    prev_wc      = prev.get("word_count", word_count)
    prev_bernays = round(len(prev_tech) / max(prev_wc, 1) * 1000, 2)

    cmp = pd.DataFrame([
        {"Run": "Letzter Run", "Orwell": prev_orwell,
         "Bernays": prev_bernays, "Techniken": len(prev_tech), "Modell": prev.get("llm_model", "-")},
        {"Run": "Dieser Run",  "Orwell": ft.get("orwell_index", "-"),
         "Bernays": bernays,   "Techniken": len(techniques),  "Modell": adapter.model},
    ]).set_index("Run")
    display(cmp)
else:
    print("06_final_result.json nicht gefunden.")